In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
from pyspark.sql import functions as F

# --- Config ---
raw_path = "/Volumes/workspace/aml_raw/landing/HI-Small_Trans.csv"
catalog = "workspace"
schema = "aml_bronze"
table_name = "raw_transactions"

# --- Explicit schema ---
trans_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("from_bank", StringType(), True),
    StructField("from_account", StringType(), True),
    StructField("to_bank", StringType(), True),
    StructField("to_account", StringType(), True),
    StructField("amount_received", DoubleType(), True),
    StructField("receiving_currency", StringType(), True),
    StructField("amount_paid", DoubleType(), True),
    StructField("payment_currency", StringType(), True),
    StructField("payment_format", StringType(), True),
    StructField("is_laundering", IntegerType(), True),
])

# --- Read raw CSV ---
df = (
    spark.read
    .option("header", "true")
    .schema(trans_schema)
    .csv(raw_path)
)

# --- Add ingestion metadata ---
df_bronze = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

# --- Create schema if not exists ---
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# --- Write as managed Delta table ---
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema}.{table_name}")
)

print(f"Loaded {df_bronze.count():,} rows into {catalog}.{schema}.{table_name}")

In [0]:
spark.sql(f"SELECT * FROM {catalog}.{schema}.{table_name} LIMIT 10").show()

spark.sql(f"""
    SELECT is_laundering, COUNT(*) as cnt
    FROM {catalog}.{schema}.{table_name}
    GROUP BY is_laundering
""").show()